# Evaluate the performance of the pre-trained model

## Environment Setup

In [1]:
!nvidia-smi

Wed Jan 15 23:40:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 551.61                 Driver Version: 551.61         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   38C    P2             39W /  320W |    1344MiB /  16376MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

if torch.cuda.is_available():
    print("CUDA is ready :)")
else:
    print("CUDA needs more work to use.")

CUDA is ready :)


In [26]:
import cv2, os, random, shutil
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedatasith/sku110k-annotations")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\randy\.cache\kagglehub\datasets\thedatasith\sku110k-annotations\versions\14


In [31]:
def draw_labels(image_path, label_path):
    # load image/labels
    image = cv2.imread(image_path)
    label = np.loadtxt(label_path, delimiter=" ", ndmin=2)
    # draw labels
    image_size = image.shape[:2]
    for c, xc, yc, w, h in label:
        xmin = xc - w/2
        ymin = yc - h/2
        xmax = xc + w/2
        ymax = yc + h/2

        xmin *= image_size[1]
        ymin *= image_size[0]
        xmax *= image_size[1]
        ymax *= image_size[0]

        start_point = (int(xmin), int(ymin))
        end_point = (int(xmax), int(ymax))
        color = (0, 255, 0)
        thickness = 10

        image = cv2.rectangle(image, start_point, end_point, color, thickness)

    return image 

In [32]:
model = YOLO("./runs/detect/train/weights/best.pt")

# draw two images, one with groundtruth labels and one with predicted labels
# set image/label path
image_path = path + "/SKU110k_fixed/images/test/test_0.jpg"
label_path = path + "/SKU110k_fixed/labels/test/test_0.txt"
# draw groundtruth labels
img_gt = draw_labels(image_path, label_path)
# draw predicted labels
# img_pred = model.predict(image_path)
#display images
# plt.figure(figsize=(10, 10))
# plt.subplot(1, 2, 1)
cv2.imwrite('output.jpg', img_gt)
plt.imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
# plt.title("Groundtruth labels")
plt.axis("off")
# plt.subplot(1, 2, 2)
# # plt.imshow(cv2.cvtColor(img_pred[0], cv2.COLOR_BGR2RGB))
# plt.title("Predicted labels")
# plt.axis("off")
plt.show()



<Figure size 640x480 with 1 Axes>

In [ ]:
# do perspective transform to get a 45 degree view of the image
def perspective_transform(image_path):

    # Read the image as cv2 object
    image = cv2.imread(image_path)

    # Get the dimensions of the image
    (h, w) = image.shape[:2]

    # Define the source points (corners of the original image)
    src = np.float32([
        [0, 0],
        [w - 1, 0],
        [0, h - 1],
        [w - 1, h - 1]
    ])

    # Define the destination points (simulate the tilt effect)
    dst_right = np.float32([
        [0, 0],             # Top-left stays in place
        [w-1, (h * 0.2)-1], # Top-right corner moves down
        [0, h],             # Bottom-left stayes in place
        [w-1, (h * 0.6)-1]  # Bottom-right corner moves up
    ])
    dst_left = np.float32([
        [0, h * 0.2],   # Top-left corner moves down
        [w-1, 0],       # Top-right corner stays in place
        [0, h * 0.6],   # Bottom-left corner moves up
        [w-1, h]        # Bottom-right corner stays in place
    ])
    # Apply the perspective transform
    M = cv2.getPerspectiveTransform(src, dst_right)

    image_transformed = []
    image_transformed.append(cv2.warpPerspective(image, cv2.getPerspectiveTransform(src, dst_right), (image.shape[1], image.shape[0])))
    image_transformed.append(cv2.warpPerspective(image, cv2.getPerspectiveTransform(src, dst_left), (image.shape[1], image.shape[0])))   


    return image_transformed

# test the perspective transform function by reading an image and display the transformed image
image_path = path+"/SKU110K_fixed/images/train/train_2.jpg"
image_transformed = perspective_transform(image_path)

for image in image_transformed:
    show_images(image)
